In [18]:
import requests
from bs4 import BeautifulSoup
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import FastEmbedEmbeddings


def scrape_page(url):
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    headings = [h.get_text().strip() for h in soup.find_all(["h1", "h2", "h3"])]
    paragraphs = [p.get_text().strip() for p in soup.find_all("p")]
    code_blocks = [code.get_text().strip() for code in soup.find_all("code")]

    return headings, paragraphs, code_blocks


def build_documents(urls):
    documents = []

    for url in urls:
        try:
            response = requests.get(url, timeout=10)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, "html.parser")

            current_section = None
            current_content = []

            for tag in soup.find_all(["h1", "h2", "h3", "p", "code"]):

                # 🔹 If it's a header → start new section
                if tag.name in ["h1", "h2", "h3"]:
                    
                    # Save previous section
                    if current_section and current_content:
                        documents.append(
                            Document(
                                page_content="\n\n".join(current_content),
                                metadata={
                                    "source": url,
                                    "section": current_section
                                }
                            )
                        )

                    # Start new section
                    current_section = tag.get_text().strip()
                    current_content = []

                # 🔹 Paragraphs
                elif tag.name == "p":
                    text = tag.get_text().strip()
                    if text:
                        current_content.append(text)

                # 🔹 Code blocks
                elif tag.name == "code":
                    code = tag.get_text().strip()
                    if code:
                        current_content.append(f"CODE:\n{code}")

            # Save last section
            if current_section and current_content:
                documents.append(
                    Document(
                        page_content="\n\n".join(current_content),
                        metadata={
                            "source": url,
                            "section": current_section
                        }
                    )
                )

            print(f"Scraped: {url}")

        except Exception as e:
            print(f"Failed to scrape {url}: {e}")

    return documents


def split_documents(documents):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=100,
        chunk_overlap=5
    )

    split_docs = splitter.split_documents(documents)
    return split_docs


def build_vectorstore(split_docs):
    embeddings = FastEmbedEmbeddings()

    vectorstore = FAISS.from_documents(
        documents=split_docs,
        embedding=embeddings
    )

    return vectorstore


if __name__ == "__main__":
    urls = [
        "https://cadquery.readthedocs.io/en/latest/apireference.html"
    ]

    # Step 1: Create documents
    documents = build_documents(urls)
    print(documents)

    # Step 2: Chunk them
    split_docs = split_documents(documents)
    # print(split_docs[0])

    # # Step 3: Build vector DB
    # vectorstore = build_vectorstore(split_docs)

    # # Save for reuse
    # vectorstore.save_local("cadquery_faiss_index")

    # print(f"Total chunks: {len(split_docs)}")

Scraped: https://cadquery.readthedocs.io/en/latest/apireference.html
[Document(metadata={'source': 'https://cadquery.readthedocs.io/en/latest/apireference.html', 'section': 'API Reference\uf0c1'}, page_content='The CadQuery API is made up of 4 main objects:\n\nSketch – Construct 2D sketches\n\nWorkplane – Wraps a topological entity and provides a 2D modelling context.\n\nSelector – Filter and select things\n\nAssembly – Combine objects into assemblies.\n\nThis page lists  methods of these objects grouped by functional area\n\nSee also\n\nThis page lists api methods grouped by functional area.\nUse CadQuery Class Summary to see methods alphabetically by class.'), Document(metadata={'source': 'https://cadquery.readthedocs.io/en/latest/apireference.html', 'section': 'Sketch initialization\uf0c1'}, page_content='Creating new sketches.\n\nSketch(parent,\xa0locs,\xa0obj)\n\nCODE:\nSketch\n\n2D sketch.\n\nSketch.importDXF(filename[,\xa0tol,\xa0exclude,\xa0...])\n\nCODE:\nSketch.importDXF\n\nI

In [2]:
docs

[{'source': 'https://cadquery.readthedocs.io/en/latest/apireference.html',
  'content': 'The CadQuery API is made up of 4 main objects:\nSketch – Construct 2D sketches\nWorkplane – Wraps a topological entity and provides a 2D modelling context.\nSelector – Filter and select things\nAssembly – Combine objects into assemblies.\nThis page lists  methods of these objects grouped by functional area\nSee also\nThis page lists api methods grouped by functional area.\nUse CadQuery Class Summary to see methods alphabetically by class.\nCreating new sketches.\nSketch(parent,\xa0locs,\xa0obj)\n2D sketch.\nSketch.importDXF(filename[,\xa0tol,\xa0exclude,\xa0...])\nImport a DXF file and construct face(s)\nWorkplane.sketch()\nInitialize and return a sketch\nSketch.finalize()\nFinish sketch construction and return the parent.\nSketch.copy()\nCreate a partial copy of the sketch.\nSketch.located(loc)\nCreate a partial copy of the sketch with a new location.\nSketch.moved(...)\nCreate a partial copy of t

In [7]:
data['text']

'The CadQuery API is made up of 4 main objects:\nSketch – Construct 2D sketches\nWorkplane – Wraps a topological entity and provides a 2D modelling context.\nSelector – Filter and select things\nAssembly – Combine objects into assemblies.\nThis page lists  methods of these objects grouped by functional area\nSee also\nThis page lists api methods grouped by functional area.\nUse CadQuery Class Summary to see methods alphabetically by class.\nCreating new sketches.\nSketch(parent,\xa0locs,\xa0obj)\n2D sketch.\nSketch.importDXF(filename[,\xa0tol,\xa0exclude,\xa0...])\nImport a DXF file and construct face(s)\nWorkplane.sketch()\nInitialize and return a sketch\nSketch.finalize()\nFinish sketch construction and return the parent.\nSketch.copy()\nCreate a partial copy of the sketch.\nSketch.located(loc)\nCreate a partial copy of the sketch with a new location.\nSketch.moved(...)\nCreate a partial copy of the sketch with moved _faces.\nSelecting, tagging and manipulating elements.\nSketch.tag(

In [5]:
data['code']

'Sketch\n\nSketch.importDXF\n\nWorkplane.sketch\n\nSketch.finalize\n\nSketch.copy\n\nSketch.located\n\nSketch.moved\n\nSketch.tag\n\nSketch.select\n\nSketch.reset\n\nSketch.delete\n\nSketch.faces\n\nSketch.edges\n\nSketch.vertices\n\nSketch.face\n\nSketch.rect\n\nSketch.circle\n\nSketch.ellipse\n\nSketch.trapezoid\n\nSketch.slot\n\nSketch.regularPolygon\n\nSketch.polygon\n\nSketch.rarray\n\nSketch.parray\n\nSketch.distribute\n\nSketch.each\n\nSketch.push\n\nSketch.hull\n\nSketch.offset\n\nSketch.fillet\n\nSketch.chamfer\n\nSketch.clean\n\nSketch.edge\n\nSketch.segment\n\nSketch.arc\n\nSketch.spline\n\nSketch.close\n\nSketch.assemble\n\nSketch.constrain\n\nSketch.solve\n\nWorkplane\n\nWorkplane.center\n\nWorkplane.lineTo\n\nWorkplane.line\n\nWorkplane.vLine\n\nWorkplane.vLineTo\n\nWorkplane.hLine\n\nWorkplane.hLineTo\n\nWorkplane.polarLine\n\nWorkplane.polarLineTo\n\nWorkplane.moveTo\n\nWorkplane.move\n\nWorkplane.spline\n\nWorkplane.parametricCurve\n\nWorkplane.parametricSurface\n\nWor